# Design de Experimentos e Analise Estatistica

Neste notebook, voce vai aprender a **planejar experimentos rigorosos** - desde o calculo
de quantos dados precisa ate a analise causal dos resultados. Cada experimento mal desenhado
desperdiça recursos e pode levar a conclusoes erradas.

**Analogia**: Imagine que voce e um detetive. Um bom detetive nao sai interrogando todo mundo
aleatoriamente - ele planeja quem interrogar, controla o ambiente e elimina suspeitos
sistematicamente. Design de experimentos e o "manual do detetive" da ciencia.

## Pre-requisitos e Fio Narrativo

| Conceito | Notebook | Por que |
|----------|----------|--------|
| Testes de Hipotese | `1_2_estatistica_inferencial` | p-value, poder, alfa/beta |
| Estatistica Bayesiana | `1_3_estatistica_bayesiana` | Priores, posteriors para A/B bayesiano |
| Regressao | `1_4_regressao_estatistica` | Modelos lineares e diagnostico |

**Fio narrativo**: Em `1_2` voce aprendeu a **testar** hipoteses. Aqui voce aprende a
**planejar** o experimento de forma que o teste seja valido. Sem design correto,
nenhum teste estatistico salva suas conclusoes.

**Tempo estimado**: 8-10 horas

## Por que Design de Experimentos em ML?

Em ML, "experimentos" acontecem o tempo todo:

- **A/B testing em producao**: qual modelo serve melhor o usuario?
- **Hyperparameter tuning**: grid search e um experimento fatorial
- **Feature selection**: testar se uma feature melhora o modelo e um teste de hipotese
- **Validacao cruzada**: e um design experimental para estimar performance

Um experimento mal desenhado pode levar a:
1. **Sem randomizacao**: comparar modelo A vs B, mas A recebe usuarios desktop e B mobile → confounding!
2. **Amostra pequena**: nao ter poder para detectar melhoria real de 0.5% em CTR
3. **Peeking**: olhar metricas todo dia e "declarar vencedor" quando p < 0.05 por acaso

Design correto envolve:
- **Randomizacao**: garante que grupos sao comparaveis
- **Replicacao**: repeticao reduz variancia
- **Controle**: isolar a variavel de interesse
- **Bloqueamento**: controlar variaveis confundidoras conhecidas

In [ ]:
import numpy as np

import matplotlib.pyplot as plt


# norm, t
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

print('=== DESIGN DE EXPERIMENTOS ===' )
print(f'\nPrincípios:' )
print(f'  1. Randomização: evita viés de seleção')
print(f'  2. Replicação: reduz variância')
print(f'  3. Controle: mede efeito de uma variável isolada')
print(f'  4. Bloqueamento: controla variáveis de confusão')


## 1. Tamanho Amostral e Poder Estatistico

**Analogia**: Imagine procurar uma moeda no chao de um estadio de futebol. Com uma lanterna
pequena (amostra pequena), voce provavelmente nao encontra. Com um holofote (amostra grande),
encontra facil. O "poder" e a probabilidade de encontrar a moeda *se ela estiver la*.

**Definicao formal**: O poder estatistico (1 - beta) e a probabilidade de rejeitar H0 quando
H0 e realmente falsa. Depende de tres fatores:
- **Effect size** (d de Cohen): o tamanho do efeito que queremos detectar
- **Alfa**: a taxa de erro tipo I que aceitamos
- **N**: o tamanho amostral

### Por que em ML?

Antes de um A/B test, voce PRECISA saber quantos usuarios precisa. Se lançar com N insuficiente,
vai desperdicar semanas de trafico e nao conseguir detectar melhorias reais.

In [ ]:
# Cálculo de tamanho amostral
from statsmodels.stats.power import tt_ind_solve_power

print('\n=== CÁLCULO DE TAMANHO AMOSTRAL ===' )

alpha = 0.05
power = 0.80
effect_size = 0.2  # Cohen's d (pequeno, médio, grande = 0.2, 0.5, 0.8)

n = tt_ind_solve_power(
    effect_size=effect_size,
    nobs1=None,
    alpha=alpha,
    power=power,
    ratio=1.0,
    alternative='two-sided'
)

print(f'\nParâmetros:')
print(f'  α (erro tipo I): {alpha}')
print(f'  Power (1-β): {power}')
print(f'  Effect size: {effect_size} (Cohen\'s d)')
print(f'\nTamanho amostral por grupo: {n:.0f}')
print(f'Total: {2*n:.0f} sujeitos')

# Efeito do tamanho amostral
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

alphas = np.linspace(0.01, 0.20, 20)
powers_01 = [tt_ind_solve_power(0.1, None, a, 0.80, alternative='two-sided') for a in alphas]
powers_05 = [tt_ind_solve_power(0.5, None, a, 0.80, alternative='two-sided') for a in alphas]
powers_08 = [tt_ind_solve_power(0.8, None, a, 0.80, alternative='two-sided') for a in alphas]

axes[0].plot(alphas, powers_01, 'o-', label='d=0.1 (pequeno)', linewidth=2)
axes[0].plot(alphas, powers_05, 's-', label='d=0.5 (médio)', linewidth=2)
axes[0].plot(alphas, powers_08, '^-', label='d=0.8 (grande)', linewidth=2)
axes[0].set_xlabel('α (nível de significância)')
axes[0].set_ylabel('Tamanho amostral por grupo')
axes[0].set_title('Tamanho Amostral vs Efeito e α')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

effects = np.linspace(0.1, 1.0, 20)
power_80 = [tt_ind_solve_power(e, None, 0.05, 0.80, alternative='two-sided') for e in effects]
power_90 = [tt_ind_solve_power(e, None, 0.05, 0.90, alternative='two-sided') for e in effects]

axes[1].plot(effects, power_80, 'o-', label='Power=0.80', linewidth=2)
axes[1].plot(effects, power_90, 's-', label='Power=0.90', linewidth=2)
axes[1].set_xlabel('Effect size (Cohen\'s d)')
axes[1].set_ylabel('Tamanho amostral por grupo')
axes[1].set_title('Tamanho Amostral vs Effect Size e Power')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### O que observar

- Note como effect size pequeno (d=0.1) exige amostra *enormemente* maior que effect size grande (d=0.8)
- O grafico log-scale mostra que a relacao e aproximadamente quadratica: para detectar efeito 2x menor, precisa de ~4x mais dados
- Aumentar power de 0.80 para 0.90 custa relativamente poucos dados extras comparado ao ganho

### O que concluir

- **Sempre calcule N antes de coletar dados**: e o erro mais comum em A/B testing nao fazer isso
- **Effect size importa mais que p-value**: um efeito minusculo pode ser "significante" com N enorme, mas irrelevante na pratica
- **Trade-off custo-beneficio**: power de 0.80 e convencao, mas em decisoes caras, use 0.90+

### Conexao com outros notebooks

- O conceito de distribuicoes amostrais de `1_2_estatistica_inferencial` e a base do calculo de poder
- Em `4_1_pipeline_ml`, o tamanho do dataset de teste segue a mesma logica: precisa de N suficiente para detectar diferencas entre modelos

## 2. A/B Testing Bayesiano

**Analogia**: No teste frequentista, voce pergunta "qual a chance de ver estes dados se nao
ha diferenca?". No bayesiano, voce pergunta diretamente "qual a probabilidade de B ser melhor
que A, dados os resultados?". E como a diferenca entre "quao improvavel e esta evidencia?"
vs "qual a probabilidade do suspeito ser culpado?".

**Definicao formal**: Usando priors Beta(alpha, beta) e atualizando com dados de conversao
(Bernoulli), obtemos posteriors analiticos. A probabilidade de B > A e calculada por simulacao
Monte Carlo dos posteriors.

### Por que em ML?

A/B bayesiano e preferido em muitas empresas de tech porque:
- Permite **early stopping** sem inflar erro tipo I
- Da uma **probabilidade direta** (ex: "93% de chance de B ser melhor")
- Incorpora **conhecimento previo** via prior

In [ ]:
print('\n=== A/B TESTING BAYESIANO ===' )
print(f'\nVantagens do Bayesian A/B:')
print(f'  - Permite stopping rule (parada prematura)')
print(f'  - Interpretação direta: P(B>A|dados)')
print(f'  - Incorpora prior knowledge')
print(f'  - Menos múltiplos testes implícitos')

# Simulação de experimento bayesiano
np.random.seed(42)

# prior_a = stats.beta(1, 1)  # Uniforme
# prior_b = stats.beta(1, 1)

n_sims = 10000
posterior_samples_a = []
posterior_samples_b = []

conversions_a = 40
trials_a = 500
conversions_b = 55
trials_b = 500

alpha_post_a = 1 + conversions_a
beta_post_a = 1 + (trials_a - conversions_a)
alpha_post_b = 1 + conversions_b
beta_post_b = 1 + (trials_b - conversions_b)

samples_a = np.random.beta(alpha_post_a, beta_post_a, n_sims)
samples_b = np.random.beta(alpha_post_b, beta_post_b, n_sims)

prob_b_better = (samples_b > samples_a).mean()
lifting = ((samples_b / samples_a) - 1) * 100

print(f'\nResultados:')
print(f'  Conversão A: {conversions_a}/{trials_a} ({conversions_a/trials_a:.1%})')
print(f'  Conversão B: {conversions_b}/{trials_b} ({conversions_b/trials_b:.1%})')
print(f'  P(B > A | dados): {prob_b_better:.1%}')
print(f'  Lift médio: {lifting.mean():.1f}% (CI95: [{np.percentile(lifting, 2.5):.1f}%, {np.percentile(lifting, 97.5):.1f}%])')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Posteriors
axes[0].hist(samples_a, bins=50, alpha=0.5, label='A', density=True)
axes[0].hist(samples_b, bins=50, alpha=0.5, label='B', density=True)
axes[0].set_xlabel('Taxa de Conversão')
axes[0].set_ylabel('Densidade')
axes[0].set_title('Distribuições Posteriores')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distribuição de lift
axes[1].hist(lifting, bins=50, alpha=0.7, edgecolor='black')
axes[1].axvline(np.percentile(lifting, 2.5), color='r', linestyle='--', label='95% CI')
axes[1].axvline(np.percentile(lifting, 97.5), color='r', linestyle='--')
axes[1].set_xlabel('Lift (%)')
axes[1].set_ylabel('Frequência')
axes[1].set_title(f'Distribuição do Lift (P(B>A)={prob_b_better:.1%})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### O que observar

- Os posteriors Beta sao distribuicoes completas, nao apenas um ponto (media). Isso captura a *incerteza*
- O lift tem uma distribuicao inteira - voce nao sabe que B e "37% melhor", sabe que o lift esta entre ~5% e ~70% com 95% de confianca
- Com 500 trials, os posteriors ja sao razoavelmente estreitos (prior uniforme tem pouco impacto)

### O que concluir

- **Bayesiano da respostas mais naturais**: "P(B>A) = 95%" e mais util que "p = 0.03" para tomada de decisao
- **Prior importa menos com mais dados**: com N grande, o prior Beta(1,1) vs Beta(10,10) faz pouca diferenca
- **Credible intervals != confidence intervals**: o bayesiano diz "95% de chance do parametro estar aqui", o frequentista diz "95% dos intervalos construidos assim conteriam o parametro"

### Conexao com outros notebooks

- A atualizacao de prior -> posterior e exatamente o Teorema de Bayes de `1_3_estatistica_bayesiana`
- A simulacao Monte Carlo conecta com os metodos de `1_1_estatistica_descritiva` (bootstrap)

## 3. Stopping Rule: Sequential Testing

**Analogia**: Imagine jogar uma moeda e decidir "se der cara 3 vezes seguidas, paro e declaro
moeda viciada". Mas voce olha depois de cada jogada. Se a moeda for justa, eventualmente vai
dar 3 caras seguidas por acaso - voce concluiu errado! Esse e o **peeking problem**.

**Definicao formal**: Cada "olhada" nos dados e um teste implicito. Se voce olha 20 vezes
com alfa=0.05, a probabilidade de pelo menos um falso positivo e 1-(1-0.05)^20 ≈ 64%!
Solucoes incluem: alpha spending functions (O'Brien-Fleming), sequential probability ratio test,
ou simplesmente pre-definir N e nao olhar ate o final.

### Por que em ML?

Em A/B testing de produtos, ha enorme pressao para "declarar vencedor" rapido. Stakeholders
querem resultados. Sem disciplina de stopping rules, voce vai "descobrir" melhorias que nao existem.

In [ ]:
print('\n=== SEQUENTIAL TESTING (Peaking Problem) ===' )
print(f'\nProblema: Olhar resultados intermediários aumenta erro tipo I')
print(f'Solução: Usar "spending function" ou parada pré-planejada')

np.random.seed(42)

# Simular acúmulo de dados
trials_per_batch = 100
num_batches = 20

group_a = np.random.binomial(1, 0.08, trials_per_batch * num_batches)
group_b = np.random.binomial(1, 0.10, trials_per_batch * num_batches)

p_values_sequential = []
conversions_a_seq = []
conversions_b_seq = []

for i in range(1, num_batches + 1):
    end_idx = i * trials_per_batch
    a_conv = group_a[:end_idx].sum()
    b_conv = group_b[:end_idx].sum()
    
    conversions_a_seq.append(a_conv / end_idx)
    conversions_b_seq.append(b_conv / end_idx)
    
    # Teste de proporções
    z = (b_conv/end_idx - a_conv/end_idx) / np.sqrt((a_conv*(end_idx-a_conv) + b_conv*(end_idx-b_conv))/end_idx**2)
    p_val = 2 * (1 - norm.cdf(abs(z)))
    p_values_sequential.append(p_val)

print(f'\nResultados:')
print(f'Final p-value: {p_values_sequential[-1]:.4f}')
print(f'Min p-value (peaking): {min(p_values_sequential):.4f}')
print(f'Risco: vários testes aumentam P(erro tipo I)')

fig, ax = plt.subplots(figsize=(12, 5))
trials_seq = np.arange(1, num_batches + 1) * trials_per_batch
ax.plot(trials_seq, p_values_sequential, 'o-', linewidth=2, markersize=6, label='P-value')
ax.axhline(0.05, color='r', linestyle='--', linewidth=2, label='α = 0.05')
ax.set_xlabel('Número de Trials')
ax.set_ylabel('P-value')
ax.set_title('Sequential Testing: P-values ao longo do tempo')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### O que observar

- O p-value oscila fortemente no inicio (poucas observacoes) e estabiliza com mais dados
- Mesmo sem efeito real (ou efeito pequeno), o p-value pode cruzar 0.05 temporariamente
- Se voce tivesse parado quando p < 0.05 pela primeira vez, teria concluido erroneamente

### O que concluir

- **Nunca declare resultado olhando intermediarios sem correcao**: se voce olha K vezes, precisa corrigir alfa
- **Pre-defina N e nao olhe antes**: a disciplina mais simples e mais eficaz
- **Sequential testing formal existe**: metodos como O'Brien-Fleming permitem "olhadas planejadas" com alfa ajustado

### Conexao com outros notebooks

- O problema de multiplos testes reaparece em `1_2_estatistica_inferencial` (correcao de Bonferroni)
- Em hyperparameter tuning (`4_2_otimizacao_hiperparametros`), early stopping de treinamento segue logica similar

## 4. A/B/C Testing (Multivariado)

**Analogia**: Comparar 2 restaurantes e facil - voce vai nos dois e decide. Mas comparar 10?
Se fizer todas as comparacoes pareadas (45 pares!), a chance de um falso positivo explode.
O teste omnibus (ANOVA/qui-quadrado) primeiro verifica "ha alguma diferenca?", so depois
investiga qual par difere.

**Definicao formal**: Para K grupos com dados categoricos, use o teste qui-quadrado de
homogeneidade. Se rejeitar H0, faca comparacoes pareadas com correcao de Bonferroni
(alfa_corrigido = alfa / n_comparacoes).

### Por que em ML?

Testar multiplas variantes de modelo, layout ou feature simultaneamente e comum. O erro e
rodar K*(K-1)/2 testes pareados sem correcao, encontrando "diferencas significantes" espurias.

In [ ]:
print('\n=== A/B/C TESTING ===' )

# Dados
groups_data = {
    'A': {'conversions': 50, 'trials': 1000},
    'B': {'conversions': 65, 'trials': 1000},
    'C': {'conversions': 55, 'trials': 1000}
}

print(f'\nDataset:')
for group, data in groups_data.items():
    ctr = data['conversions'] / data['trials']
    print(f'  Grupo {group}: {data["conversions"]}/{data["trials"]} ({ctr:.1%})')

# ANOVA para testes de proporção
from scipy.stats import chi2_contingency

contingency = np.array([
    [groups_data[g]['conversions'], groups_data[g]['trials'] - groups_data[g]['conversions']]
    for g in ['A', 'B', 'C']
])

chi2, p_anova, dof, expected = chi2_contingency(contingency)

print(f'\nANOVA (Qui-Quadrado):')
print(f'  χ² = {chi2:.3f}, p-value = {p_anova:.4f}')
print(f'  Diferença significante? {"Sim" if p_anova < 0.05 else "Não"}')

# Comparações pareadas
from scipy.stats import proportions_ztest

print(f'\nComparações Pareadas:')
for i, g1 in enumerate(['A', 'B', 'C']):
    for g2 in ['A', 'B', 'C'][i+1:]:
        count = [groups_data[g1]['conversions'], groups_data[g2]['conversions']]
        nobs = [groups_data[g1]['trials'], groups_data[g2]['trials']]
        z, p = proportions_ztest(count, nobs)
        print(f'  {g1} vs {g2}: p = {p:.4f}')


### O que observar

- O teste omnibus (qui-quadrado) da um unico p-value para "existe alguma diferenca entre os grupos?"
- As comparacoes pareadas mostram *onde* esta a diferenca, mas precisam de correcao
- Com 3 grupos, sao 3 comparacoes. Com 10 grupos, seriam 45! O risco de falso positivo escala rapidamente

### O que concluir

- **Sempre faca teste omnibus antes de comparacoes pareadas**: evita inflacao de erro tipo I
- **Use correcao de Bonferroni ou Holm**: alfa_corrigido = 0.05 / n_comparacoes
- **Mais variantes = mais dados necessarios**: cada grupo adicional dilui o tamanho amostral

### Conexao com outros notebooks

- O teste qui-quadrado e a ANOVA de `1_2_estatistica_inferencial` sao a base destes testes
- Em `4_2_otimizacao_hiperparametros`, comparar multiplas configuracoes e exatamente este cenario

## 5. Causalidade: DAGs e Confounding

**Analogia**: Vendas de sorvete e afogamentos estao correlacionados. Proibir sorvete
reduziria afogamentos? Claro que nao! O calor (confunder) causa ambos. DAGs (Directed Acyclic
Graphs) sao mapas que mostram "quem causa quem" e revelam quando uma correlacao e espuria.

**Definicao formal**: Um DAG e um grafo direcionado sem ciclos onde:
- **Confunder** (Z -> X, Z -> Y): cria correlacao espuria. CONTROLAR Z remove a espuriedade
- **Mediador** (X -> Z -> Y): Z transmite o efeito. NAO controlar se quer efeito total
- **Collider** (X -> Z <- Y): X e Y sao independentes. Controlar Z CRIA correlacao espuria (vies de selecao!)

### Por que em ML?

Modelos preditivos nao precisam de causalidade (apenas correlacao). Mas se voce quer **intervir**
(ex: "se mudarmos X, Y muda?"), precisa de causalidade. Isso e critico em:
- Recomendacao: "mostrar este produto CAUSA mais compras?"
- Medicina: "este tratamento CAUSA melhora?"
- Politicas: 'esta feature CAUSA mais engajamento?'

In [ ]:
print('\n=== CAUSALIDADE vs CORRELAÇÃO ===' )
print(f'\nCorrelação ≠ Causalidade!')
print(f'\nTrês tipos de relacionamento:')
print(f'  1. X → Y (X causa Y) - efeito causal')
print(f'  2. Z → X e Z → Y (confounding) - X e Y correlacionados, mas Z é causa comum')
print(f'  3. X → Z → Y (mediação) - X causa Y indiretamente')

print(f'\nExemplo: Correlação entre sapatos grandes e leitura')
print(f'  - Confunding: Idade afeta ambas (crianças maiores leem melhor)')
print(f'  - Solução: Controlar por idade ou usar randomização')

print(f'\nDAG (Directed Acyclic Graph):')
print(f'  Z → X → Y: X é mediador (não controlar por X se interesse em efeito total)')
print(f'  Z → X ← Y: Z é colidador (controlar por Z introduz viés!)')
print(f'  Z → X, Z → Y: Z é confunder (SEMPRE controlar)')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Exemplo 1: Confounding
ax = axes[0]
ax.text(0.5, 0.8, 'Confounding', ha='center', fontsize=12, fontweight='bold')
ax.arrow(0.3, 0.6, -0.15, -0.15, head_width=0.05, fc='black', ec='black')
ax.arrow(0.3, 0.6, 0.15, -0.15, head_width=0.05, fc='black', ec='black')
ax.text(0.3, 0.65, 'Z (Confunder)', ha='center', fontsize=10)
ax.text(0.15, 0.35, 'X', ha='center', fontsize=10)
ax.text(0.55, 0.35, 'Y', ha='center', fontsize=10)
ax.arrow(0.2, 0.3, 0.3, 0, head_width=0.05, fc='red', ec='red', linestyle='--')
ax.text(0.35, 0.1, 'Correlação Espúria', ha='center', fontsize=9, color='red')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

# Exemplo 2: Mediação
ax = axes[1]
ax.text(0.5, 0.8, 'Mediação', ha='center', fontsize=12, fontweight='bold')
ax.arrow(0.15, 0.6, 0.2, -0.15, head_width=0.05, fc='black', ec='black')
ax.arrow(0.45, 0.45, 0.2, -0.15, head_width=0.05, fc='black', ec='black')
ax.text(0.1, 0.65, 'X', ha='center', fontsize=10)
ax.text(0.5, 0.5, 'Z (Mediador)', ha='center', fontsize=10)
ax.text(0.8, 0.3, 'Y', ha='center', fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

# Exemplo 3: Collider
ax = axes[2]
ax.text(0.5, 0.8, 'Collider (viés se controlar!)', ha='center', fontsize=12, fontweight='bold')
ax.arrow(0.2, 0.6, 0.15, -0.15, head_width=0.05, fc='black', ec='black')
ax.arrow(0.8, 0.6, -0.15, -0.15, head_width=0.05, fc='black', ec='black')
ax.text(0.15, 0.65, 'X', ha='center', fontsize=10)
ax.text(0.85, 0.65, 'Y', ha='center', fontsize=10)
ax.text(0.5, 0.35, 'Z (Collider)', ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

plt.tight_layout()
plt.show()

### O que observar

- O diagrama de Confounding mostra que a correlacao X-Y e *espuria* (causada por Z)
- No Collider, X e Y sao independentes... ate voce controlar por Z! Isso e contraintuitivo
- A diferenca entre mediacao e confounding esta na direcao das setas, nao na estrutura visivel

### O que concluir

- **Sempre desenhe o DAG antes de escolher variaveis de controle**: controlar a variavel errada pode *piorar* o vies
- **Collider bias e o erro mais sutil**: controlar por uma variavel que parece "relevante" pode criar correlacao onde nao havia
- **Randomizacao resolve confounding, mas nao e sempre possivel**: em dados observacionais, use tecnicas como propensity score matching

### Conexao com outros notebooks

- Confounding conecta diretamente com multicolinearidade em `1_4_regressao_estatistica` (VIF)
- Em `5_3_interpretabilidade_modelos`, a diferenca entre importancia correlacional e causal e fundamental

## 6. Cross-Validation como Design Experimental

**Analogia**: Cross-validation e como um professor que faz 5 provas diferentes para avaliar
um aluno, cada vez com questoes diferentes. A nota media e mais confiavel que uma unica prova.
Os principios de design experimental (randomizacao, replicacao) estao todos presentes.

**Definicao formal**: K-fold CV divide os dados em K partes, treina em K-1 e testa em 1,
repetindo K vezes. E um design experimental completo:
- **Randomizacao**: dados sao embaralhados antes da divisao
- **Replicacao**: K folds = K "experimentos"
- **Controle**: cada fold usa o mesmo pipeline de treino

### Por que em ML?

CV e O metodo padrao para estimar performance de modelos. Mas ha sutilezas:
- **Stratified** K-fold preserva proporcao de classes
- **Time Series** split respeita ordem temporal (sem data leakage)
- **Group** K-fold mantem grupos intactos (ex: mesmo paciente nao em treino e teste)

In [ ]:
print('\n=== CROSS-VALIDATION EM ML ===' )
print(f'\nK-fold CV é um tipo de "experimento" estadístico:')
print(f'  - Randomização: divisão aleatória dos dados')
print(f'  - Replicação: K folds = K "experimentos" independentes')
print(f'  - Variância reduzida: média de K resultados')

from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit

# Dados
n_samples = 100
n_classes = 2
X = np.random.randn(n_samples, 10)
y = np.random.binomial(1, 0.5, n_samples)

print(f'\nTipos de Cross-Validation:')
print(f'\n1. K-Fold (padrão): divisão aleatória')
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_sizes = []
for train_idx, test_idx in kf.split(X, y):
    fold_sizes.append((len(train_idx), len(test_idx)))
print(f'   Fold sizes (train, test): {fold_sizes[:2]}')

print(f'\n2. Time Series CV: respecta ordem temporal')
tscv = TimeSeriesSplit(n_splits=5)
fold_times = []
for train_idx, test_idx in tscv.split(X):
    fold_times.append((train_idx[-1], test_idx[0]))
print(f'   Train-test indices (últimos train, primeiros test): {fold_times[:2]}')

print(f'\n3. Stratified K-Fold: preserva distribuição de classes')
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f'   Acurácia por fold: {scores.round(3)}')
print(f'   Média: {scores.mean():.3f} ± {scores.std():.3f}')


### O que observar

- K-fold e Stratified K-fold embaralham dados - adequados para dados i.i.d.
- Time Series split NUNCA usa dados futuros para treinar - note como train cresce e test avanca
- O desvio padrao entre folds indica estabilidade: se um fold e muito diferente, investigue

### O que concluir

- **Escolha o CV certo para seu problema**: dados temporais DEVEM usar TimeSeriesSplit, nunca KFold
- **Report mean +/- std**: apenas a media esconde instabilidade entre folds
- **CV nao e magic bullet**: se os dados tem leakage (ex: feature derivada do target), CV nao detecta

### Conexao com outros notebooks

- CV como estimador de performance reaparece em todo `4_1_pipeline_ml` e `4_2_otimizacao_hiperparametros`
- O conceito de vies-variancia do CV (K pequeno = alto vies, K grande = alta variancia) conecta com `1_4_regressao_estatistica`

## 7. Exercicios Praticos

### Exercicio 1: Calculo de Tamanho Amostral

Voce e data scientist de um e-commerce e quer testar uma nova pagina de checkout.
A taxa de conversao atual e 3.2%. Voce espera que a nova pagina melhore para 3.8%.
Calcule:
1. O effect size (Cohen's h para proporcoes)
2. O tamanho amostral necessario (power=0.85, alpha=0.05)
3. Quantos dias de trafico voce precisa se tem 10.000 visitantes/dia?

In [ ]:
# TODO: Exercicio 1 - Calculo de Tamanho Amostral
# Dados
p_controle = 0.032
p_tratamento = 0.038
power_desejado = 0.85
alpha = 0.05
visitantes_dia = 10000

# 1. Calcule Cohen's h: h = 2 * arcsin(sqrt(p1)) - 2 * arcsin(sqrt(p2))
cohens_h = None  # TODO

# 2. Calcule N por grupo usando tt_ind_solve_power (aproximacao)
n_por_grupo = None  # TODO

# 3. Calcule dias necessarios
dias_necessarios = None  # TODO

print(f"Cohen's h: {cohens_h}")
print(f"N por grupo: {n_por_grupo}")
print(f"Dias necessarios: {dias_necessarios}")

In [ ]:
# SOLUCAO - Exercicio 1
from statsmodels.stats.power import tt_ind_solve_power

p_controle = 0.032
p_tratamento = 0.038
power_desejado = 0.85
alpha = 0.05
visitantes_dia = 10000

# 1. Cohen's h para proporcoes
cohens_h = 2 * np.arcsin(np.sqrt(p_tratamento)) - 2 * np.arcsin(np.sqrt(p_controle))
print(f"Cohen's h: {cohens_h:.4f}")
print(f"Interpretacao: efeito {'pequeno' if abs(cohens_h) < 0.3 else 'medio' if abs(cohens_h) < 0.5 else 'grande'}")

# 2. N por grupo (usando aproximacao normal)
n_por_grupo = tt_ind_solve_power(
    effect_size=cohens_h,
    nobs1=None,
    alpha=alpha,
    power=power_desejado,
    ratio=1.0,
    alternative='two-sided'
)
print(f"\nN por grupo: {n_por_grupo:.0f}")
print(f"N total: {2*n_por_grupo:.0f}")

# 3. Dias necessarios (metade do trafico para cada grupo)
n_total = 2 * n_por_grupo
dias_necessarios = np.ceil(n_total / visitantes_dia)
print(f"\nDias necessarios: {dias_necessarios:.0f} dias")
print(f"(com {visitantes_dia:,} visitantes/dia, {visitantes_dia//2:,} por grupo)")

### Exercicio 2: A/B Test Bayesiano Completo

Voce rodou um A/B test e coletou os seguintes dados:
- Controle: 1200 conversoes em 15000 trials
- Tratamento: 1350 conversoes em 15000 trials

Implemente a analise bayesiana completa:
1. Defina priors (use Beta(1,1) uniforme)
2. Calcule posteriors
3. Estime P(B > A) via Monte Carlo
4. Calcule o lift esperado e seu intervalo de 95%

In [ ]:
# TODO: Exercicio 2 - A/B Test Bayesiano
conv_a, trials_a = 1200, 15000
conv_b, trials_b = 1350, 15000

# 1. Prior: Beta(1, 1)
prior_alpha, prior_beta = 1, 1

# 2. Posterior parameters
alpha_post_a = None  # TODO: prior_alpha + conv_a
beta_post_a = None   # TODO: prior_beta + (trials_a - conv_a)
alpha_post_b = None  # TODO
beta_post_b = None   # TODO

# 3. Monte Carlo: sample 50000 from each posterior
n_mc = 50000
# samples_a = None  # TODO: np.random.beta(...)
# samples_b = None  # TODO
# prob_b_better = None  # TODO: (samples_b > samples_a).mean()

# 4. Lift
# lift = None  # TODO: (samples_b / samples_a - 1) * 100

print(f"P(B > A): {prob_b_better}")
print(f"Lift medio: {lift.mean():.2f}%")

In [ ]:
# SOLUCAO - Exercicio 2
np.random.seed(42)
conv_a, trials_a = 1200, 15000
conv_b, trials_b = 1350, 15000

# 1. Prior
prior_alpha, prior_beta = 1, 1

# 2. Posteriors
alpha_post_a = prior_alpha + conv_a
beta_post_a = prior_beta + (trials_a - conv_a)
alpha_post_b = prior_alpha + conv_b
beta_post_b = prior_beta + (trials_b - conv_b)

print(f"Posterior A: Beta({alpha_post_a}, {beta_post_a})")
print(f"Posterior B: Beta({alpha_post_b}, {beta_post_b})")

# 3. Monte Carlo
n_mc = 50000
samples_a = np.random.beta(alpha_post_a, beta_post_a, n_mc)
samples_b = np.random.beta(alpha_post_b, beta_post_b, n_mc)
prob_b_better = (samples_b > samples_a).mean()

print(f"\nP(B > A | dados): {prob_b_better:.4f} ({prob_b_better:.1%})")

# 4. Lift
lift = (samples_b / samples_a - 1) * 100
print(f"Lift medio: {lift.mean():.2f}%")
print(f"Lift 95% CI: [{np.percentile(lift, 2.5):.2f}%, {np.percentile(lift, 97.5):.2f}%]")

# Decisao
if prob_b_better > 0.95:
    print("\nDecisao: IMPLEMENTAR B (>95% de confianca)")
elif prob_b_better > 0.90:
    print("\nDecisao: Evidencia forte, considerar implementar B")
else:
    print("\nDecisao: Continuar coletando dados")

### Exercicio 3: Identificacao de Confounders com DAG

Considere o cenario: voce quer medir se "tempo de estudo" (X) causa "nota na prova" (Y).
Variaveis disponiveis: motivacao (Z1), horas de sono (Z2), dificuldade da prova (Z3).

1. Desenhe o DAG (como texto) indicando as relacoes causais
2. Identifique quais variaveis sao confounders, mediadoras ou colliders
3. Determine quais variaveis controlar na regressao

In [ ]:
# TODO: Exercicio 3 - Analise de DAG
# Preencha as relacoes causais e classifique cada variavel

# 1. DAG (descreva as setas)
dag = {
    'Z1 (motivacao)': None,  # TODO: lista de variaveis que Z1 causa
    'Z2 (sono)': None,       # TODO
    'Z3 (dificuldade)': None, # TODO
    'X (estudo)': None,       # TODO
    'Y (nota)': None           # TODO: [] (outcome)
}

# 2. Classificacao
confounders = None    # TODO: lista de variaveis confundidoras
mediadores = None     # TODO: lista de mediadores
colliders = None      # TODO: lista de colliders

# 3. Variaveis a controlar
controlar = None      # TODO: lista de variaveis para incluir na regressao

print(f"Confounders: {confounders}")
print(f"Mediadores: {mediadores}")
print(f"Colliders: {colliders}")
print(f"Controlar na regressao: {controlar}")

In [ ]:
# SOLUCAO - Exercicio 3
print("=== DAG: Tempo de Estudo -> Nota ===")
print()
print("Relacoes causais:")
print("  Z1 (motivacao) -> X (estudo)    [motivacao aumenta tempo de estudo]")
print("  Z1 (motivacao) -> Y (nota)      [motivacao melhora nota diretamente]")
print("  Z2 (sono) -> X (estudo)         [sono afeta capacidade de estudar]")
print("  Z2 (sono) -> Y (nota)           [sono afeta desempenho na prova]")
print("  Z3 (dificuldade) -> Y (nota)    [dificuldade afeta nota]")
print("  X (estudo) -> Y (nota)          [estudo melhora nota]")
print()

# Classificacao
confounders = ['Z1 (motivacao)', 'Z2 (sono)']
mediadores = []   # Nenhuma variavel e mediadora neste DAG
colliders = []    # Nenhuma variavel e collider neste DAG

print("Classificacao:")
print(f"  Confounders: {confounders}")
print(f"  -> Z1 causa tanto X quanto Y (backdoor path: X <- Z1 -> Y)")
print(f"  -> Z2 causa tanto X quanto Y (backdoor path: X <- Z2 -> Y)")
print(f"  Mediadores: nenhum")
print(f"  Colliders: nenhum")
print()

controlar = ['Z1 (motivacao)', 'Z2 (sono)']
print("Variaveis a controlar:")
print(f"  {controlar}")
print("  -> Bloqueiam os backdoor paths, isolando o efeito causal de X em Y")
print()
print("NAO controlar Z3 (dificuldade):")
print("  -> Z3 so afeta Y, nao X. Nao e confunder.")
print("  -> Controlar Z3 reduz variancia residual (bom para precisao),")
print("  -> mas nao e necessario para eliminar vies causal.")

### O que observar nos exercicios

- O Exercicio 1 revela que efeitos pequenos em proporcoes (3.2% vs 3.8%) exigem amostras enormes - milhares de usuarios
- O Exercicio 2 mostra que a analise bayesiana da uma resposta direta e acionavel ("93% de chance de B ser melhor")
- O Exercicio 3 demonstra que a escolha de variaveis de controle depende da *estrutura causal*, nao da correlacao

### O que concluir dos exercicios

- **Planejamento amostral e indispensavel**: sem ele, voce pode rodar um A/B test por semanas sem conclusao
- **Bayesiano e frequentista sao complementares**: bayesiano para decisao, frequentista para rigor
- **DAGs sao a ferramenta certa para decidir o que controlar**: intuicao falha em cenarios com colliders

### Conexao com outros notebooks

- O calculo de Cohen's h do Exercicio 1 e uma variante do effect size de `1_2_estatistica_inferencial`
- A simulacao Monte Carlo do Exercicio 2 reutiliza a logica de `1_3_estatistica_bayesiana`

### O que observar no panorama geral

- Design de experimentos e o *unico* framework que permite afirmacoes causais (junto com DAGs)
- A progressao natural e: planejar N -> coletar dados -> testar hipotese -> verificar causalidade
- Todos os principios (randomizacao, replicacao, controle, bloqueamento) aparecem em CV de ML

### O que concluir do panorama geral

- **Sem randomizacao, nao ha causalidade**: dados observacionais so dao correlacao (com raras excecoes)
- **O framework experimental unifica estatistica classica e ML**: CV, A/B testing e hyperparameter search sao todos experimentos
- **Investir em design economiza em analise**: um experimento bem planejado precisa de analise simples; um mal planejado nao tem salvacao

### Conexao com outros notebooks

- Todo o pipeline de `4_1_pipeline_ml` e um experimento: split, treino, avaliacao seguem principios de design
- A interpretabilidade de `5_3_interpretabilidade_modelos` depende de saber se as relacoes sao causais ou correlacionais

### O que observar na relacao entre frequentista e bayesiano

- O teste frequentista (secao 3-4) responde "quao improvavel sao estes dados sob H0?"
- O teste bayesiano (secao 2) responde "qual a probabilidade de B ser melhor que A?"
- Ambos usam os mesmos dados, mas fazem perguntas fundamentalmente diferentes

### O que concluir da complementaridade

- **Use frequentista quando precisa de rigor regulatorio** (FDA, papers academicos)
- **Use bayesiano quando precisa de decisao rapida** (A/B testing em produto)
- A escolha nao e "qual e melhor", mas "qual responde a pergunta certa"

### Por que em ML?

Em MLOps, A/B testing bayesiano permite decidir mais rapido se um novo modelo e melhor,
economizando trafico e tempo. Mas para publicar resultados, reviewers esperam p-values.

### Conexao com outros notebooks

- A dualidade frequentista/bayesiano e o tema central de `1_2` vs `1_3`

### O que observar sobre validade interna vs externa

- Validade interna: o experimento mede o que diz medir? (controlou confounders?)
- Validade externa: o resultado generaliza? (outras populacoes, contextos, epocas?)
- Um experimento pode ter alta validade interna e baixa externa (laboratorio puro)

### O que concluir sobre generalizacao

- **Replicacao e o padrao ouro**: um resultado so e robusto se replica em contextos diferentes
- **Populacao do experimento != populacao de interesse**: usuarios mobile vs desktop, Brasil vs EUA
- **Sazonalidade importa**: um A/B test no Natal pode nao valer no resto do ano

### Por que em ML?

Modelos treinados em dados de uma distribuicao falham em outra (distribution shift). Validacao
cruzada mede validade interna; testes em dados realmente novos medem validade externa.

### Conexao com outros notebooks

- Distribution shift e o tema de `4_4_monitoramento_modelos` e conecta com validade externa

## 8. Erros Comuns e Armadilhas

### Erro 1: Nao calcular tamanho amostral ANTES do experimento
Coletar dados, rodar o teste, obter p = 0.06, e decidir "vou coletar mais um pouco" e **invalido**.
O tamanho amostral deve ser fixado antes. Senao, voce esta fazendo sequential testing sem correcao.

### Erro 2: Confundir poder estatistico com significancia
Alpha (tipo I) controla falsos positivos. Poder (1 - beta) controla falsos negativos. Sao
coisas diferentes! Um teste com poder baixo pode nao detectar um efeito real - e voce
conclui "nao ha efeito" quando na verdade nao teve dados suficientes.

### Erro 3: Espiar resultados intermediarios (peeking)
Olhar os resultados a cada 10 observacoes e parar quando p < 0.05 infla dramaticamente o
erro tipo I. Com 20 "olhadas", a chance de um falso positivo e ~64%. Use N pre-planejado
ou sequential testing formal (O'Brien-Fleming).

### Erro 4: Ignorar confounding e declarar causalidade
"Usuarios que clicam no botao A compram mais" nao significa que o botao A *causa* mais compras.
Pode ser que usuarios mais engajados clicam mais E compram mais (confounding). Sem
randomizacao ou controle de confounders via DAG, voce tem correlacao, nao causalidade.

### Erro 5: Controlar por collider (vies de selecao)
Condicionar em uma variavel que e *efeito* de X e Y cria correlacao espuria. Exemplo
classico: entre candidatos *aceitos* em uma universidade, notas e esportes parecem
negativamente correlacionados (compensam um ao outro para admissao). Mas na populacao geral, nao ha correlacao.

### Erro 6: Comparacoes multiplas sem correcao
Se voce testa 20 metricas em um A/B test com alpha=0.05, espera-se 1 falso positivo.
Use correcao de Bonferroni (alpha/n) ou FDR (Benjamini-Hochberg) para multiplas comparacoes.

### Erro 7: Generalizar alem do escopo do experimento
Seu A/B test funcionou com usuarios de desktop no Brasil, durante o verao. Nao assuma que
o resultado vale para mobile, outros paises ou outras epocas. Considere sempre a
**validade externa** do experimento.

## 9. Resumo e Conexoes

### Hierarquia de Conceitos

```
PERGUNTA CIENTIFICA
    |
    v
DESIGN DO EXPERIMENTO
    |
    |---> Randomizacao: remove vies de selecao
    |---> Replicacao: reduz variancia
    |---> Controle: isola variavel de interesse
    |---> Bloqueamento: controla confounders conhecidos
    |
    v
CALCULO DE TAMANHO AMOSTRAL
    |---> Effect size (Cohen's d ou h)
    |---> Alpha (erro tipo I): tipicamente 0.05
    |---> Power (1-beta): tipicamente 0.80-0.90
    |---> N = f(effect_size, alpha, power)
    |
    v
COLETA E ANALISE
    |
    |---> Frequentista: p-value, IC, teste de hipotese
    |---> Bayesiano: P(B>A|dados), posteriors, credible intervals
    |---> Multivariado: ANOVA/qui-quadrado + comparacoes pareadas
    |
    v
CAUSALIDADE
    |
    |---> DAGs: mapear relacoes causais
    |---> Confounders: CONTROLAR (bloqueiam backdoor paths)
    |---> Mediadores: NAO controlar (se quer efeito total)
    |---> Colliders: NUNCA controlar (cria vies!)
    |
    v
VALIDACAO
    |---> Validade interna: conclusao valida no seu contexto?
    |---> Validade externa: generaliza para outros contextos?
    |---> Cross-validation: design experimental para ML
```

### Tabela de Conexoes

| Conceito | Notebook anterior | Notebook futuro |
|----------|------------------|-----------------|
| p-value e testes | `1_2_estatistica_inferencial` | `4_2_otimizacao` (comparar modelos) |
| Bayesiano | `1_3_estatistica_bayesiana` | `4_3_bayesian_optimization` |
| Regressao e confounders | `1_4_regressao_estatistica` | `5_3_interpretabilidade` |
| Cross-validation | `1_2` (bootstrap) | `4_1_pipeline_ml` |
| Multiplos testes | `1_2` (Bonferroni) | `3_1_feature_engineering` (selecao) |
| Tamanho amostral | `1_1` (CLT) | `4_1` (split treino/teste) |

### Checklist de Competencias

- [ ] Sei calcular tamanho amostral com power analysis
- [ ] Sei implementar A/B testing bayesiano com posteriors Beta
- [ ] Entendo o peeking problem e sei quando usar sequential testing
- [ ] Sei fazer teste omnibus antes de comparacoes pareadas
- [ ] Sei desenhar DAGs e identificar confounders, mediadores e colliders
- [ ] Sei escolher o tipo certo de cross-validation para cada problema
- [ ] Sei distinguir causalidade de correlacao em analises

### Proximos Passos

1. **`2_1_algebra_linear_fundamentos`**: Vetores e matrizes como base para ML
2. **`3_1_feature_engineering`**: Criar e selecionar features (requer poder estatistico)
3. **`4_1_pipeline_ml`**: O pipeline completo que usa CV como design experimental